<a href="https://colab.research.google.com/github/VitorTardivo21/Redes-Neurais-e-IA-Aplicada/blob/main/04_treinamento_avaliacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Etapa 4 - Treinamento e Avaliacao do Classificador
Projeto: Diario Oficial Inteligente de Avare

Atividade 1 - Modelo ClassificadorDiario (src/model.py)
Atividade 2 - Loop de treinamento com curva de aprendizado
Atividade 3 - Avaliacao com k-fold estratificado (5 folds)
"""

import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from src.dataset import DiarioDataset, tokenizar
from src.model   import ClassificadorDiario

# ==========================================
# SEMENTE GLOBAL
# ==========================================

RANDOM_STATE = 42

def fixar_seed(seed: int = RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

fixar_seed()

# ==========================================
# CAMINHOS
# ==========================================

BASE_DIR       = os.path.dirname(os.path.abspath(__file__))
CSV_BASE       = os.path.join(BASE_DIR, "data", "processed", "base_textual.csv")
VOCAB_JSON     = os.path.join(BASE_DIR, "data", "processed", "vocab.json")
LABEL_MAP_JSON = os.path.join(BASE_DIR, "data", "processed", "label_map.json")
MODELO_PT      = os.path.join(BASE_DIR, "models", "modelo.pt")
CURVA_PNG      = os.path.join(BASE_DIR, "docs",   "curva_treinamento.png")
CONFUSAO_PNG   = os.path.join(BASE_DIR, "docs",   "matriz_confusao.png")

os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)

# ==========================================
# HIPERPARAMETROS
# ==========================================

EMBED_DIM  = 64
EPOCAS     = 40
BATCH_SIZE = 16
LR         = 0.001
MAX_LEN    = 738
N_FOLDS    = 5

# ==========================================
# CARREGA BASE COMPLETA (211 registros)
# ==========================================

print("=" * 60)
print("ETAPA 4 - Treinamento do Classificador")
print("=" * 60)

df = pd.read_csv(CSV_BASE, encoding="utf-8-sig")
df = df[df["texto"].notna() & (df["texto"].str.strip() != "")].reset_index(drop=True)

print(f"\nBase completa : {len(df)} registros | {df['rotulo'].nunique()} classes")
print("\nDistribuicao:")
for c, n in df["rotulo"].value_counts().items():
    print(f"  {c:25s}: {n:3d} registros")

# ==========================================
# RECONSTROI VOCABULARIO A PARTIR DA BASE COMPLETA
# ==========================================

print("\nReconstruindo vocabulario com base em todos os registros...")

contador = Counter()
for texto in df["texto"]:
    contador.update(tokenizar(str(texto)))

vocab = {"<PAD>": 0, "<UNK>": 1}
filtrados = 0
for token, freq in contador.most_common():
    if freq >= 2:
        vocab[token] = len(vocab)
    else:
        filtrados += 1

with open(VOCAB_JSON, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f"Vocab: {len(vocab)} tokens ({filtrados} hapax filtrados) -> salvo em vocab.json")

# Label map (mantido igual: ordem alfabetica)
with open(LABEL_MAP_JSON, encoding="utf-8") as f:
    label2id = json.load(f)

id2label = {i: c for c, i in label2id.items()}
nomes    = [id2label[i] for i in sorted(id2label)]

df["rotulo_id"] = df["rotulo"].map(label2id)

# ==========================================
# CLASS WEIGHTS — balanceia classes desiguais
# ==========================================

contagem     = df["rotulo_id"].value_counts().sort_index()
pesos        = len(df) / (len(nomes) * contagem.values.astype(float))
class_weights = torch.tensor(pesos, dtype=torch.float)

print(f"\nClass weights: { {id2label[i]: f'{w:.2f}' for i, w in enumerate(pesos)} }")

# ==========================================
# FUNCAO DE TREINO POR FOLD
# ==========================================

def treinar_fold(treino_df, teste_df, vocab, label2id, seed):
    fixar_seed(seed)

    treino_ds     = DiarioDataset(treino_df, vocab, label2id, MAX_LEN)
    teste_ds      = DiarioDataset(teste_df,  vocab, label2id, MAX_LEN)
    treino_loader = DataLoader(treino_ds, batch_size=BATCH_SIZE, shuffle=True)
    teste_loader  = DataLoader(teste_ds,  batch_size=BATCH_SIZE, shuffle=False)

    modelo     = ClassificadorDiario(len(vocab), len(label2id), EMBED_DIM)
    criterio   = nn.CrossEntropyLoss(weight=class_weights)
    otimizador = optim.Adam(modelo.parameters(), lr=LR)

    perdas = []
    for _ in range(EPOCAS):
        modelo.train()
        total = 0.0
        for x, y in treino_loader:
            otimizador.zero_grad()
            perda = criterio(modelo(x), y)
            perda.backward()
            otimizador.step()
            total += perda.item()
        perdas.append(total / len(treino_loader))

    modelo.eval()
    prevs, reais = [], []
    with torch.no_grad():
        for x, y in teste_loader:
            pred = modelo(x).argmax(dim=1)
            prevs.extend(pred.tolist())
            reais.extend(y.tolist())

    acc = sum(p == r for p, r in zip(prevs, reais)) / len(reais)
    return acc, perdas, modelo, prevs, reais

# ==========================================
# K-FOLD ESTRATIFICADO
# ==========================================

print(f"\n{'='*60}")
print(f"AVALIACAO K-FOLD ESTRATIFICADO ({N_FOLDS} folds, {EPOCAS} epocas cada)")
print(f"{'='*60}")

skf             = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
acuracias       = []
historico_folds = []
melhor_acc      = 0.0
melhor_modelo   = None
todos_reais     = []
todos_prevs     = []

for fold, (idx_treino, idx_teste) in enumerate(skf.split(df, df["rotulo_id"])):
    treino_df_f = df.iloc[idx_treino]
    teste_df_f  = df.iloc[idx_teste]

    acc, perdas, modelo_fold, prevs, reais = treinar_fold(
        treino_df_f, teste_df_f, vocab, label2id, seed=RANDOM_STATE + fold
    )

    acuracias.append(acc)
    historico_folds.append(perdas)
    todos_reais.extend(reais)
    todos_prevs.extend(prevs)

    if acc > melhor_acc:
        melhor_acc    = acc
        melhor_modelo = modelo_fold

    print(f"  Fold {fold+1}/{N_FOLDS} | treino={len(treino_df_f):3d} | teste={len(teste_df_f):2d} | acuracia={acc:.2%}")

media_acc = float(np.mean(acuracias))
std_acc   = float(np.std(acuracias))

print(f"\n{'='*60}")
print(f"RESULTADO K-FOLD")
print(f"{'='*60}")
print(f"  Acuracia por fold : {[f'{a:.2%}' for a in acuracias]}")
print(f"  Media             : {media_acc:.2%}")
print(f"  Desvio padrao     : {std_acc:.2%}")
print(f"  Melhor fold       : {max(acuracias):.2%}")

# ==========================================
# TREINO FINAL COM TODOS OS DADOS
# ==========================================

print(f"\n{'='*60}")
print(f"TREINO FINAL (todos os {len(df)} registros)")
print(f"{'='*60}")

fixar_seed(RANDOM_STATE)
ds_completo     = DiarioDataset(df, vocab, label2id, MAX_LEN)
loader_completo = DataLoader(ds_completo, batch_size=BATCH_SIZE, shuffle=True)

modelo_final = ClassificadorDiario(len(vocab), len(label2id), EMBED_DIM)
criterio_f   = nn.CrossEntropyLoss(weight=class_weights)
otimizador_f = optim.Adam(modelo_final.parameters(), lr=LR)

historico_final = []
for epoca in range(EPOCAS):
    modelo_final.train()
    total = 0.0
    for x, y in loader_completo:
        otimizador_f.zero_grad()
        perda = criterio_f(modelo_final(x), y)
        perda.backward()
        otimizador_f.step()
        total += perda.item()
    media = total / len(loader_completo)
    historico_final.append(media)
    if (epoca + 1) % 10 == 0 or epoca == 0:
        print(f"  Epoca {epoca+1:2d}/{EPOCAS} | perda: {media:.4f}")

torch.save(modelo_final.state_dict(), MODELO_PT)
print(f"\nModelo salvo: {MODELO_PT}")

total_params = sum(p.numel() for p in modelo_final.parameters())
print(f"Parametros  : {total_params:,}")

# ==========================================
# CURVA DE APRENDIZADO
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for i, perdas in enumerate(historico_folds):
    axes[0].plot(range(1, EPOCAS + 1), perdas, alpha=0.6, linewidth=1.2, label=f"Fold {i+1}")
axes[0].set_title(f"Curva de aprendizado — {N_FOLDS} folds")
axes[0].set_xlabel("Epoca"); axes[0].set_ylabel("Perda (CrossEntropy)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCAS + 1), historico_final, color="#2E75B6", linewidth=2)
axes[1].set_title("Treino final — base completa")
axes[1].set_xlabel("Epoca"); axes[1].set_ylabel("Perda (CrossEntropy)")
axes[1].grid(alpha=0.3)

plt.suptitle(f"Acuracia k-fold: {media_acc:.2%} +/- {std_acc:.2%}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CURVA_PNG, dpi=120)
plt.close()
print(f"Curva salva: {CURVA_PNG}")

# ==========================================
# AVALIACAO CONSOLIDADA (out-of-fold)
# ==========================================

print(f"\n{'='*60}")
print(f"AVALIACAO CONSOLIDADA (out-of-fold, {len(todos_reais)} predicoes)")
print(f"{'='*60}")

acc_oof = sum(p == r for p, r in zip(todos_prevs, todos_reais)) / len(todos_reais)
print(f"\nAcuracia OOF: {acc_oof:.2%}")

print("\nRelatorio por classe:")
print(classification_report(todos_reais, todos_prevs, target_names=nomes, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    todos_reais, todos_prevs, display_labels=nomes,
    cmap="Blues", ax=ax, xticks_rotation=35
)
plt.title(f"Matriz de confusao — {N_FOLDS}-fold (211 registros)\n"
          f"Acuracia: {media_acc:.2%} +/- {std_acc:.2%}")
plt.tight_layout()
plt.savefig(CONFUSAO_PNG, dpi=120)
plt.close()
print(f"Matriz salva: {CONFUSAO_PNG}")

erros = [(id2label[r], id2label[p]) for r, p in zip(todos_reais, todos_prevs) if r != p]
print(f"\nErros totais: {len(erros)} de {len(todos_reais)}")
if erros:
    print("Detalhes dos erros:")
    for real, prev in erros:
        print(f"  Real: {real:25s} -> Previsto: {prev}")

# ==========================================
# RELATORIO FINAL
# ==========================================

print("\n" + "=" * 60)
print("ETAPA 4 - CONCLUIDA")
print("=" * 60)
print(f"Base usada        : base_textual.csv ({len(df)} registros)")
print(f"Acuracia k-fold   : {media_acc:.2%} +/- {std_acc:.2%}")
print(f"Acuracia OOF      : {acc_oof:.2%}  ({len(todos_reais)} predicoes)")
print(f"Modelo salvo      : {MODELO_PT}")
print(f"Vocab atualizado  : {len(vocab)} tokens")
print(f"\nHiperparametros:")
print(f"  embed_dim  = {EMBED_DIM}")
print(f"  hidden_dim = 128")
print(f"  dropout    = 0.3")
print(f"  epocas     = {EPOCAS}")
print(f"  lr         = {LR}")
print(f"  batch_size = {BATCH_SIZE}")
print(f"  max_len    = {MAX_LEN}")
print(f"  n_folds    = {N_FOLDS}")
print(f"  class_weights = sim (balanceia decreto:116 vs edital_concurso:9)")
